# Ch 7. LLM으로 KG 구축 — 엔티티·관계 추출 (hands-on)

지식 그래프와 GraphRAG 스터디 · [study-graphrag](https://desty.github.io/study-graphrag/)

[Ch 7](https://desty.github.io/study-graphrag/part3/07-llm-extraction/) 본문 코드. **API 키가 없어도 그대로 돈다** — 키가 있으면 실제 추출, 없으면 예시 결과로 폴백한 뒤 networkx 그래프에 적재·질의까지 실행한다.

In [ ]:
!pip -q install anthropic networkx
import os, json

# 온톨로지를 가드레일로 — 허용 타입 밖은 만들지 않게 한다
ONTOLOGY = {"entities": ["Drug", "Disease", "Ingredient"],
            "relations": ["treats", "contains", "interactsWith"]}

PROMPT = '''다음 텍스트에서 트리플을 추출해 JSON으로만 답하라.
허용 엔티티 타입: {entities}
허용 관계 타입: {relations}
규칙: 허용 목록에 없는 타입은 만들지 말 것. 불확실하면 빼라.
형식: {{"triples": [{{"subj":"...","subj_type":"...","rel":"...","obj":"...","obj_type":"..."}}]}}

텍스트:
{text}'''

FALLBACK = {"triples": [
    {"subj": "아스피린", "subj_type": "Drug", "rel": "treats", "obj": "두통", "obj_type": "Disease"},
    {"subj": "아스피린", "subj_type": "Drug", "rel": "contains", "obj": "아세틸살리실산", "obj_type": "Ingredient"},
]}

def extract(text):
    try:
        import anthropic
        client = anthropic.Anthropic()  # ANTHROPIC_API_KEY 필요
        msg = client.messages.create(
            model="claude-haiku-4-5", max_tokens=1024,
            messages=[{"role": "user", "content": PROMPT.format(
                entities=ONTOLOGY["entities"], relations=ONTOLOGY["relations"], text=text)}])
        return json.loads(msg.content[0].text)
    except Exception as e:
        print(f"[키 없음/실패 → 예시 결과 사용] {type(e).__name__}")
        return FALLBACK

out = extract("아스피린은 두통을 완화하며, 성분으로 아세틸살리실산을 포함한다.")
print(json.dumps(out, ensure_ascii=False, indent=2))

## 추출한 트리플을 그래프에 적재하고 바로 질의

In [ ]:
import networkx as nx
G = nx.MultiDiGraph()
for t in out["triples"]:
    G.add_node(t["subj"], label=t["subj_type"])
    G.add_node(t["obj"], label=t["obj_type"])
    G.add_edge(t["subj"], t["obj"], rel=t["rel"])

# "아스피린이 치료하는 질병은?"
print([v for _, v, d in G.out_edges("아스피린", data=True) if d["rel"] == "treats"])
# "아스피린의 성분은?"
print([v for _, v, d in G.out_edges("아스피린", data=True) if d["rel"] == "contains"])

## 연습

1. 가드레일(허용 타입)을 빼고 `extract`를 돌리면 관계 이름이 얼마나 제멋대로 늘어나는지 비교하라 (키가 있을 때).
2. `FALLBACK`을 본인 도메인 문장으로 바꿔, 추출 없이도 적재·질의 파이프라인을 연습하라.
3. 출처 문장을 엣지 속성 `source`로 남기도록 적재 루프를 고쳐라.